# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [26]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
import os, json, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../../05_src/.secrets')  
print("OPENAI_API_KEY present?", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY present? True


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://www.newyorker.com/magazine/2004/07/26/what-is-noise"

html = requests.get(URL, timeout=60).text
soup = BeautifulSoup(html, "html.parser")

paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
article_text = "\n\n".join(paragraphs)

print(article_text[:800])
len(article_text)






© 2025 Condé Nast. All rights reserved. The New Yorker may earn a portion of sales from products that are purchased through our site as part of our Affiliate Partnerships with retailers. The material on this site may not be reproduced, distributed, transmitted, cached or otherwise used, except with the prior written permission of Condé Nast. Ad Choices


358

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
chunks = splitter.split_text(article_text)
len(chunks), chunks[0][:300]


(1,
 '© 2025 Condé Nast. All rights reserved. The New Yorker may earn a portion of sales from products that are purchased through our site as part of our Affiliate Partnerships with retailers. The material on this site may not be reproduced, distributed, transmitted, cached or otherwise used, except with ')

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from openai import OpenAI

class SummaryRecord(BaseModel):
    author: str = Field(description="Document author")
    title: str = Field(description="Document title")
    publication: str = Field(description="Publisher or outlet")
    main_summary: str = Field(description="Concise 200–300 word summary in plain English")
    key_points: List[str] = Field(description="5–8 bullet points covering core ideas")
    notable_quotes: List[str] = Field(description="Two short direct quotes, 10–25 words each")
    word_count: int = Field(description="Word count of main_summary")
    confidence: float = Field(ge=0, le=1, description="Self-rated confidence 0–1")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

SYSTEM_PROMPT = (
    "You are a careful analyst. Read the provided text chunks from Alex Ross’s article "
    "'What Is Noise?' (The New Yorker, 2004) and produce a STRICT JSON object ONLY (no markdown) "
    "that matches the given schema. Ensure author/title/publication are correct, "
    "main_summary is 200–300 words, include 5–8 key points and 2 short quotes. "
    "Return valid JSON with all fields. IMPORTANT: `confidence` must be a numeric float (0–1)."
)

def summarize_chunks(chunks, model="gpt-4o-mini"):  
    joined = "\n\n".join(chunks[:6])  # first few chunks 
    user_prompt = (
        "TEXT:\n" + joined +
        "\n\nEmit JSON ONLY with fields: author, title, publication, main_summary, key_points, notable_quotes, word_count, confidence."
    )
    resp = client.responses.create(
        model=model,
        input=[
            {"role":"system", "content":SYSTEM_PROMPT},
            {"role":"user",   "content":user_prompt}
        ],
        temperature=0.2,
    )
    return resp.output_text  # raw JSON string 

raw_json = summarize_chunks(chunks)
print(raw_json[:600])


{
  "author": "Alex Ross",
  "title": "What Is Noise?",
  "publication": "The New Yorker",
  "main_summary": "In 'What Is Noise?', Alex Ross explores the concept of noise in music and its broader implications in society. He delves into how noise has been perceived historically, from its association with chaos to its role in contemporary art and culture. Ross argues that noise is not merely a disruptive force but can also be a powerful medium for expression and innovation. He discusses various artists and composers who have embraced noise, transforming it into a legitimate form of art. The arti


In [32]:
from pydantic import ValidationError

CONF_MAP = {"high": 0.9, "medium": 0.6, "low": 0.3}

def coerce_confidence(obj):
    if isinstance(obj, dict) and "confidence" in obj:
        val = obj["confidence"]
        if isinstance(val, str):
            obj["confidence"] = CONF_MAP.get(val.strip().lower(), 0.5)
    return obj

def parse_summary_robust(raw_json: str) -> SummaryRecord:
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw_json, flags=re.DOTALL)
        if not m:
            raise
        data = json.loads(m.group(0))
    data = coerce_confidence(data)
    return SummaryRecord(**data)

try:
    result = parse_summary_robust(raw_json)
    result
except ValidationError as e:
    print("Validation error:", e)
    print("Raw text was:")
    print(raw_json)


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [33]:
import re
def word_count(s:str): 
    return len(re.findall(r"\w+", s))

wc = word_count(result.main_summary)
length_score = 1.0 if 200 <= wc <= 300 else max(0.0, 1 - abs(wc-250)/250)
wc, length_score


(128, 0.512)

In [34]:
# Simple coverage heuristic over themes relevant to the article
themes = [
    "noise","music","modernism","john cage","technology","experimental",
    "culture","sound","audience","tradition","boundaries","aesthetics"
]
def coverage_score(text, keywords):
    tl = text.lower()
    hits = sum(1 for k in keywords if k in tl)
    return hits / len(keywords)

cov = coverage_score(result.main_summary, themes)
cov


0.3333333333333333

In [35]:
from rouge_score import rouge_scorer

reference = (
    "Alex Ross surveys how twentieth-century and contemporary composers have blurred the line between noise and music, "
    "expanding the palette of acceptable sounds through technology, avant-garde experimentation, and shifting cultural attitudes. "
    "From early modernists to John Cage and beyond, he shows how listeners’ expectations change as artists reframe noise as music, "
    "arguing that the definition of music is historically contingent and continually evolving."
)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = scorer.score(reference, result.main_summary)
rougeL_f = scores['rougeL'].fmeasure
rougeL_f


0.17708333333333334

In [36]:
overall = 0.4*rougeL_f + 0.35*cov + 0.25*length_score
overall

0.3155

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [37]:
from pprint import pprint

report = {
    "Author": result.author,
    "Title": result.title,
    "Publication": result.publication,
    "Summary_Words": wc,
    "Heuristics": {
        "LengthScore(200-300)": round(length_score, 3),
        "CoverageScore": round(cov, 3),
        "ROUGE-L(f1)": round(rougeL_f, 3),
        "Overall": round(overall, 3),
    },
    "KeyPoints": result.key_points,
    "NotableQuotes": result.notable_quotes,
    "Confidence": result.confidence,
}
pprint(report)

{'Author': 'Alex Ross',
 'Confidence': 0.95,
 'Heuristics': {'CoverageScore': 0.333,
                'LengthScore(200-300)': 0.512,
                'Overall': 0.316,
                'ROUGE-L(f1)': 0.177},
 'KeyPoints': ['Noise has historically been viewed as chaotic and disruptive.',
               'Contemporary artists utilize noise as a legitimate form of '
               'expression.',
               'The article examines the tension between traditional music and '
               'avant-garde movements.',
               'Noise challenges conventional notions of beauty and harmony.',
               'Ross highlights various composers who have embraced noise in '
               'their work.',
               'Noise serves as a metaphor for societal dissonance.',
               'The exploration of noise reflects the complexities of modern '
               'life.'],
 'NotableQuotes': ['Noise is not merely a disruptive force but can also be a '
                   'powerful medium for expre

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
